# Demo — Fine-Grained PHI Access Control with AWS Lake Formation
In Veridian Health's Medallion Lakehouse, `silver_patient_master` contains direct HIPAA/GDPR identifiers (`full_name`, `mrn`, `email`, `phone`). Rather than duplicating tables for each team, **AWS Lake Formation** enforces column-level, row-level (`DataCellsFilter`), and tag-based (`LF-TBAC`) policies on the single Silver table.

In this demo, we grant `vh_clinical_researcher` column-level access to `silver_patient_master` excluding all four PHI columns, verify enforcement via `query_as_persona()`, and inspect how the underlying S3 -> Glue -> Lake Formation chain is provisioned in Terraform.

## Step 0: Connect to your Cloud Lab

Run the cell below to configure your AWS Cloud Lab environment and seed the **Veridian Health Medallion Lakehouse** (`veridian_lakehouse` in AWS Glue, S3 `bronze/silver/gold` Parquet prefixes, and AWS Lake Formation enforcement) via `setup_aws.py`.

In [1]:
import os
os.environ["VERIDIAN_LF_LIVE"] = "1"
os.environ["VERIDIAN_AWS_REGION"] = "us-east-1"
# If your Vocareum AWS credentials are not already in the environment, paste them below:
# os.environ["AWS_ACCESS_KEY_ID"] = "..."
# os.environ["AWS_SECRET_ACCESS_KEY"] = "..."
# os.environ["AWS_SESSION_TOKEN"] = "..."

import boto3
import setup_aws
glue = boto3.client("glue", region_name=os.environ["VERIDIAN_AWS_REGION"])
try:
    glue.get_database(Name="veridian_lakehouse")
    print("Glue database 'veridian_lakehouse' is already seeded and ready.")
except glue.exceptions.EntityNotFoundException:
    setup_aws.main()

Glue database 'veridian_lakehouse' is already seeded and ready.


In [2]:
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
from demo_toolkit import connect_lakehouse, lh_backend, HEALTHCARE_PERSONAS

lh = connect_lakehouse()
print(f"Backend: {lh_backend()} -> {type(lh).__name__} (Glue DB: {lh.database}, Workgroup: {lh.athena_workgroup})")
for role, arn in lh.personas.items():
    print(f"  {role:30} -> {arn}")

Backend: live -> AwsMedallionLakehouse (Glue DB: veridian_lakehouse, Workgroup: primary)
  vh_clinical_researcher         -> arn:aws:iam::123456789012:role/vh_clinical_researcher
  vh_eu_care_coordinator         -> arn:aws:iam::123456789012:role/vh_eu_care_coordinator
  vh_him_steward                 -> arn:aws:iam::123456789012:role/vh_him_steward
  vh_hipaa_compliance_auditor    -> arn:aws:iam::123456789012:role/vh_hipaa_compliance_auditor


## 1. Grant column-level PHI masking to `vh_clinical_researcher`

In [3]:
lh.grant_column_subset(
    "vh_clinical_researcher",
    "silver_patient_master",
    exclude=["full_name", "mrn", "email", "phone"],
)
display(lh.grants_summary())

                       persona                  table                  access
0       vh_clinical_researcher  silver_patient_master  cols=6 of 10; rows=all
1       vh_eu_care_coordinator                 (none)               no grants
2               vh_him_steward                 (none)               no grants
3  vh_hipaa_compliance_auditor                 (none)               no grants


## 2. Verify enforcement by querying as `vh_clinical_researcher`

In [4]:
df = lh.query_as_persona("vh_clinical_researcher", "SELECT * FROM silver_patient_master LIMIT 5")
print("Visible columns for vh_clinical_researcher:", list(df.columns))
display(df)

Visible columns for vh_clinical_researcher: ['mpi_id', 'jurisdiction', 'country', 'plan_tier', 'hipaa_research_auth', 'enrolled_ts']


   mpi_id jurisdiction country        plan_tier  hipaa_research_auth          enrolled_ts
0    1001    CA-PIPEDA      CA   commercial_ppo                    1  2024-11-23T12:00:00
1    1002    CA-PIPEDA      CA  provincial_plus                    1  2024-08-24T19:00:00
2    1003     US-HIPAA      US  provincial_plus                    0  2023-08-12T08:00:00
3    1004     US-HIPAA      US  provincial_plus                    1  2023-07-02T04:00:00
4    1005     US-HIPAA      US   commercial_ppo                    1  2023-11-08T02:00:00


## 3. How the Medallion governance chain is provisioned in Terraform (IaC)

In [5]:
from pathlib import Path
print(Path("terraform/main.tf").read_text())

# =============================================================================
# Veridian Health — Medallion Lakehouse Governance Chain in Terraform (IaC)
# Storage (S3) -> Catalog (AWS Glue) -> Access Control (AWS Lake Formation)
# =============================================================================

terraform {
  required_version = ">= 1.5.0"
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

variable "aws_region" {
  type    = string
  default = "us-east-1"
}

variable "lakehouse_database" {
  type    = string
  default = "veridian_lakehouse"
}

data "aws_caller_identity" "current" {}

# 1. Storage Layer — Medallion S3 Bucket (Bronze / Silver / Gold prefixes)
resource "aws_s3_bucket" "medallion_lakehouse" {
  bucket = "${data.aws_caller_identity.current.account_id}-veridian-lakehouse"
}

resource "aws_s3_bucket_server_side_encryption_configuration" "phi_encryption" {
  bucket = aws_s3_bucket.medallion_lakehouse.id
  rul